In [ ]:
# Aplicação Estratégica — Ranking de Municípios de Risco

Responde às perguntas de negócio do desafio: quais municípios apresentam
maior risco educacional, e como prever municípios que podem não atingir
metas futuras. Usamos o modelo treinado (Random Forest) aplicado ao
conjunto de teste (fora da amostra de treino) para gerar probabilidades
previstas de alfabetização por município, comparadas com a meta oficial

In [1]:
import pandas as pd
ranking = pd.read_csv("../reports/municipio_risk_ranking.csv")
ranking.shape

(1055, 8)

In [ ]:
## Duas lentes de risco: gap pra meta vs. risco absoluto

Ao olhar os 15 piores casos por `gap_previsto_vs_meta`, um padrão chamou
atenção: **10 de 15 são do Rio Grande do Sul** — um estado geralmente
forte em educação. Isso não é um erro do modelo; é uma consequência de
como a métrica de gap é definida.

O RS tem metas muito ambiciosas (75-80%, entre as mais altas do país).
Mesmo com uma probabilidade prevista moderada (47-54%, nem catastrófica),
o gap contra uma meta de 80% fica enorme. Já municípios do Maranhão,
Bahia e Piauí aparecem com probabilidade prevista genuinamente baixa
(0,44-0,52) — risco real, não efeito de meta ambiciosa.

**Conclusão prática:** um gestor público precisa das duas lentes.
"Gap pra meta" prioriza onde o esforço extra é necessário pra cumprir o
que aquele território já se comprometeu a entregar. "Risco absoluto"
prioriza onde a situação é mais grave em termos absolutos, independente
da meta local. Um painel de política pública deveria mostrar as duas,
não escolher uma.

In [2]:
print("Top 10 por RISCO ABSOLUTO (menor probabilidade prevista, independente da meta):")
print(ranking.nsmallest(10, "prob_media_prevista")[["id_municipio", "sigla_uf", "prob_media_prevista", "meta", "gap_previsto_vs_meta"]].to_string(index=False))

Top 10 por RISCO ABSOLUTO (menor probabilidade prevista, independente da meta):
 id_municipio sigla_uf  prob_media_prevista  meta  gap_previsto_vs_meta
      2912608       BA             0.274965 27.17              0.326506
      2924108       BA             0.292250 27.85              1.374973
      2906808       BA             0.294147 36.80             -7.385253
      2922250       BA             0.294415 38.02             -8.578482
      2900207       BA             0.295816 35.44             -5.858433
      2926103       BA             0.301455 25.46              4.685539
      2906303       BA             0.308167 36.12             -5.303290
      2906824       BA             0.310114 25.86              5.151408
      2928000       BA             0.313970 32.30             -0.902976
      2918100       BA             0.315751 34.24             -2.664850


In [ ]:
### Conclusão: as duas lentes contam histórias diferentes — e a segunda é mais grave

Os 10 municípios de maior **risco absoluto** são **todos da Bahia** — uma
concentração muito mais clara e acionável do que o ranking por gap, que
misturava RS (metas ambiciosas) com casos de risco real.

Mais grave: em alguns desses municípios baianos, a meta oficial é tão
baixa (25-38%) que uma probabilidade prevista de ~30% — ainda uma situação
crítica em termos absolutos — aparece com gap **positivo** (bate a própria
meta). Isso expõe um risco de política pública que vai além do modelo:
metas municipais mal calibradas podem mascarar situações de crise real
atrás de um indicador "verde".

**Recomendação prática:** priorizar intervenção por risco absoluto nos
municípios baianos identificados, e revisar a calibração de metas
municipais que estejam desproporcionalmente baixas em relação à gravidade
real da situação.